# Siedler AI - Resource Reward Training

**Reward:** Nur am Ende basierend auf gesammelten Taler + Schwefel

Potential Scharfschützen = min(Taler/250, Schwefel/70)

## Setup

In [ ]:
# 1. GPU prüfen
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Keine GPU! Gehe zu: Runtime -> Change runtime type -> GPU")

In [ ]:
# 2. Dependencies installieren
!pip install -q gymnasium stable-baselines3 sb3-contrib

In [ ]:
# 3. ZIP-Datei hochladen (NUR EINE DATEI!)
from google.colab import files
import zipfile
import os

print("Bitte 'colab_files.zip' hochladen:")
uploaded = files.upload()

# ZIP entpacken
with zipfile.ZipFile("colab_files.zip", 'r') as zip_ref:
    zip_ref.extractall(".")
print("Dateien entpackt!")

# Config-Ordner erstellen
os.makedirs("config", exist_ok=True)
if os.path.exists("game_data.json") and not os.path.exists("config/game_data.json"):
    import shutil
    shutil.move("game_data.json", "config/game_data.json")
    print("game_data.json -> config/")

## Training

In [ ]:
import time
import numpy as np
import torch as th
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

from environment import SiedlerScharfschuetzenEnv
from multihead_policy import MultiHeadMaskablePolicy

# Kosten für Scharfschützen
SCHARFSCHUETZEN_TALER_COST = 250
SCHARFSCHUETZEN_SCHWEFEL_COST = 70


class ResourceRewardEnv(SiedlerScharfschuetzenEnv):
    """Environment mit Reward nur am Ende basierend auf Ressourcen."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.total_taler_earned = 0.0
        self.total_schwefel_earned = 0.0
        self.start_taler = 0.0
        self.start_schwefel = 0.0
        self.peak_taler = 0.0
        self.peak_schwefel = 0.0

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self.start_taler = float(self.resources.get("Taler", 0))
        self.start_schwefel = float(self.resources.get("Schwefel", 0))
        self.peak_taler = self.start_taler
        self.peak_schwefel = self.start_schwefel
        self.total_taler_earned = 0.0
        self.total_schwefel_earned = 0.0
        return obs, info

    def step(self, action):
        obs, _, terminated, truncated, info = super().step(action)

        current_taler = float(self.resources.get("Taler", 0))
        current_schwefel = float(self.resources.get("Schwefel", 0))

        if current_taler > self.peak_taler:
            self.total_taler_earned += (current_taler - self.peak_taler)
            self.peak_taler = current_taler
        if current_schwefel > self.peak_schwefel:
            self.total_schwefel_earned += (current_schwefel - self.peak_schwefel)
            self.peak_schwefel = current_schwefel

        reward = 0.0
        if terminated or truncated:
            total_taler = self.start_taler + self.total_taler_earned
            total_schwefel = self.start_schwefel + self.total_schwefel_earned
            reward = min(
                total_taler / SCHARFSCHUETZEN_TALER_COST,
                total_schwefel / SCHARFSCHUETZEN_SCHWEFEL_COST
            )

        return obs, reward, terminated, truncated, info


class ProgressCallback(BaseCallback):
    def __init__(self, check_freq=5000):
        super().__init__()
        self.check_freq = check_freq
        self.best_potential = 0.0

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            print(f"Step {self.n_calls:,}")
        return True


def make_env(rank, seed=0):
    def _init():
        env = ResourceRewardEnv(player_id=1, use_spatial_obs=False)
        env.reset(seed=seed + rank)
        return env
    return _init


print("Setup complete!")

In [ ]:
# TRAINING CONFIG
N_ENVS = 8          # Parallele Environments
TIMESTEPS = 500_000  # Training Steps

print("=" * 60)
print("SIEDLER AI - RESOURCE REWARD TRAINING")
print("=" * 60)
print(f"Environments: {N_ENVS}")
print(f"Timesteps: {TIMESTEPS:,}")
print(f"Device: {'cuda' if th.cuda.is_available() else 'cpu'}")
print("=" * 60)

# Environment erstellen
env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])

# Policy-Kwargs
head_sizes = env.env_method("get_action_head_sizes")[0]
phase_dim = env.get_attr("phase_dim")[0]

policy_kwargs = {
    "net_arch": [512, 256, 256],
    "action_head_sizes": head_sizes,
    "phase_dim": phase_dim,
}

# Modell erstellen
model = MaskablePPO(
    MultiHeadMaskablePolicy,
    env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,
    ent_coef=0.02,
    policy_kwargs=policy_kwargs,
    device="cuda" if th.cuda.is_available() else "cpu",
    verbose=0,
)

print("\nModell erstellt, starte Training...")

In [ ]:
# TRAINING
start_time = time.time()

model.learn(
    total_timesteps=TIMESTEPS,
    callback=ProgressCallback(check_freq=10000),
    progress_bar=True,
)

train_time = time.time() - start_time
print(f"\nTraining abgeschlossen in {train_time:.1f}s")
print(f"Steps/Sekunde: {TIMESTEPS / train_time:.0f}")

# Modell speichern
model.save("siedler_resource_reward")
print("\nModell gespeichert: siedler_resource_reward.zip")

## Evaluation

In [ ]:
# EVALUATION
print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

eval_env = ResourceRewardEnv(player_id=1, use_spatial_obs=False)

for ep in range(3):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0

    while not done:
        action_mask = eval_env.get_action_mask()
        action, _ = model.predict(obs, deterministic=True, action_masks=action_mask)
        if hasattr(action, 'item'):
            action = action.item()
        obs, reward, terminated, truncated, _ = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated

    total_taler = eval_env.start_taler + eval_env.total_taler_earned
    total_schwefel = eval_env.start_schwefel + eval_env.total_schwefel_earned
    potential = min(total_taler / 250, total_schwefel / 70)

    print(f"\nEpisode {ep+1}:")
    print(f"  Taler: {total_taler:.0f} (verdient: +{eval_env.total_taler_earned:.0f})")
    print(f"  Schwefel: {total_schwefel:.0f} (verdient: +{eval_env.total_schwefel_earned:.0f})")
    print(f"  Potential Scharfschützen: {potential:.2f}")
    print(f"  Reward: {total_reward:.2f}")

In [ ]:
# Download Modell
from google.colab import files
files.download("siedler_resource_reward.zip")